# RF-Diffusion Reproduction (Colab)

Open this notebook in Google Colab and select `Runtime -> Run all`.

It will:

1. Clone the official `RF-Diffusion` repo at the fixed commit.
2. Download the official dataset and pretrained models.
3. Run the official Wi-Fi inference (`inference.py --task_id 0`).
4. Run the official 5G FDD inference (`inference.py --task_id 2`).
5. Run a small-scale training (4 blocks, hidden 64, 200 iters).
6. Run efficiency sweeps.
7. Aggregate metrics and plot figures.

All artifacts are saved under `/content/results/`.

In [ ]:
import os, sys, subprocess, time, json, pathlib

ROOT = pathlib.Path('/content')
UP = ROOT / 'RF-Diffusion'
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
for sub in ['raw', 'logs', 'metrics']:
    (RESULTS / sub).mkdir(exist_ok=True)

!nvidia-smi -L || print('No GPU')
!python --version

In [ ]:
# 1. Clone official repo at pinned commit.
if not UP.exists():
    !git clone https://github.com/mobicom24/RF-Diffusion.git {UP}
%cd {UP}
!git checkout eb872b0c4543da65424f5598ae40826e76e7edea
%cd /content

In [ ]:
# 2. Download dataset + models.
%cd {UP}
!wget -q https://github.com/mobicom24/RF-Diffusion/releases/download/dataset_model/dataset.zip
!wget -q https://github.com/mobicom24/RF-Diffusion/releases/download/dataset_model/model.zip
!unzip -q -o dataset.zip
!unzip -q -o model.zip
!mkdir -p dataset && mv wifi dataset/wifi && mv fmcw dataset/fmcw && mv mimo dataset/mimo
!rm -f dataset.zip model.zip
%cd /content

In [ ]:
# 3. Install dependencies.
!pip install -q torch torchvision numpy scipy tensorboard tqdm matplotlib \
                  pytorch_fid pytorch-msssim psutil pyyaml

In [ ]:
# 4. Wi-Fi official inference.
import subprocess, time
log_path = RESULTS / 'logs' / 'wifi_colab.log'
t0 = time.time()
with open(log_path, 'w') as f:
    subprocess.run(['python', 'inference.py', '--task_id', '0'],
                   cwd=str(UP), stdout=f, stderr=subprocess.STDOUT, check=False)
print(f'Wi-Fi done in {time.time() - t0:.1f}s, log: {log_path}')
!tail -5 {log_path}

In [ ]:
# 5. 5G FDD official inference.
log_path = RESULTS / 'logs' / 'mimo_colab.log'
t0 = time.time()
with open(log_path, 'w') as f:
    subprocess.run(['python', 'inference.py', '--task_id', '2'],
                   cwd=str(UP), stdout=f, stderr=subprocess.STDOUT, check=False)
print(f'5G done in {time.time() - t0:.1f}s, log: {log_path}')
!tail -5 {log_path}

In [ ]:
# 6. Small-scale training (reduced).
import sys, pathlib
PROJECT = pathlib.Path('/content/project')
if not PROJECT.exists():
    !git clone https://github.com/yourname/project {PROJECT} 2>/dev/null || echo 'No project repo; skip'
# Upload the project zip from the local machine via Colab files panel if available.
# Otherwise run the small-train script directly.
print('Upload the project scripts/run_small_train.py if needed, or run the cell below.')

In [ ]:
# 7. Aggregate final metrics.
import re, json, csv
wifi_log = (RESULTS / 'logs' / 'wifi_colab.log').read_text(errors='ignore')
mimo_log = (RESULTS / 'logs' / 'mimo_colab.log').read_text(errors='ignore')
ssim = re.search(r'Average SSIM: ([0-9.]+)', wifi_log)
fid  = re.search(r'FID value: ([0-9.]+)', wifi_log)
snr  = re.search(r'Average SNR: ([0-9.]+)', mimo_log)
summary = {
    'wifi_ssim': float(ssim.group(1)) if ssim else None,
    'wifi_fid': float(fid.group(1)) if fid else None,
    'mimo_snr_db': float(snr.group(1).rstrip('.')) if snr else None,
}
print(json.dumps(summary, indent=2))
with open(RESULTS / 'metrics' / 'colab_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

## Notes

- If you see CUDNN errors, just rerun the cell.
- If a download fails, manually upload `dataset.zip` and `model.zip` to `/content/`.
- All raw outputs are saved under `/content/RF-Diffusion/dataset/.../output/`.
- For full efficiency sweeps, see `scripts/run_efficiency*.py` in the project repo.